# EX: Explainable AI with SHAP

In this exercise, we will simulate a predictive maintenance model for an aircraft engine. We will train a simple model and then use the shap library (an XAI tool) to mathematically prove why the model made a specific prediction, ensuring operator trust.

Note: You must install the required packages (pip install shap scikit-learn numpy pandas) before running this code.


In [ ]:
# Only run this cell after downloading and selecting your kernel
!python.exe -m pip install --upgrade pip
!pip install shap

In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import shap

# 1. Simulate Aircraft Engine Sensor Data (100 flights)
# Features: Vibration (Hz), Temp (C), Pressure (PSI)
np.random.seed(42)
X = pd.DataFrame({
    'Vibration': np.random.normal(50, 10, 100),
    'Temperature': np.random.normal(800, 50, 100),
    'Pressure': np.random.normal(120, 15, 100)
})

# Target: Engine Wear Score (Higher is worse). 
# Math: Vibration strongly increases wear. Temp slightly increases wear.
y = (X['Vibration'] * 2.5) + (X['Temperature'] * 0.5) - (X['Pressure'] * 0.2) + np.random.normal(0, 10, 100)

# 2. Train a "Black Box" Model (Random Forest)
print("Training Black Box Model...")
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X, y)

# 3. Analyze a specific flight (Flight 0)
flight_idx = 0
flight_data = X.iloc[[flight_idx]]
prediction = model.predict(flight_data)[0]
print(f"\nModel Prediction for Flight 0 Wear Score: {prediction:.2f}")

# 4. Apply Explainable AI (SHAP)
print("\nCalculating SHAP values to explain the prediction...")
# We create an explainer object using our trained model

explainer = shap.TreeExplainer(model)

# We calculate the SHAP values for Flight 0
shap_values = explainer.shap_values(flight_data)

# The expected_value is the baseline average prediction across all flights
baseline = explainer.expected_value[0]

print(f"\nBaseline Average Wear Score: {baseline:.2f}")
print("--- How each sensor contributed to the final prediction ---")

# Print the mathematical contribution of each feature
for feature_name, shap_value in zip(X.columns, shap_values[0]):
    direction = "increased" if shap_value > 0 else "decreased"
    print(f"{feature_name}: {direction} the score by {abs(shap_value):.2f}")

# Mathematically prove the explanation:
# Baseline + Sum(SHAP Values) = Final Prediction
math_check = baseline + np.sum(shap_values[0])
print(f"\nMath Check (Baseline + SHAP sum): {math_check:.2f} == Prediction: {prediction:.2f}")


/Users/mtodd/471Book-1/.book_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training Black Box Model...

Model Prediction for Flight 0 Wear Score: 470.80

Calculating SHAP values to explain the prediction...

Baseline Average Wear Score: 499.89
--- How each sensor contributed to the final prediction ---
Vibration: increased the score by 6.84
Temperature: decreased the score by 34.95
Pressure: decreased the score by 0.98

Math Check (Baseline + SHAP sum): 470.80 == Prediction: 470.80



## Interpreting the Results

When you run this script, the model outputs a raw prediction for the engine wear. However, the XAI shap library breaks that number down. It proves mathematically that the model isn't just guessing; it shows the operator exactly how much the high Vibration reading drove the prediction up, allowing the maintenance crew to trust the AI and target their repairs efficiently.
